In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit import segment_od_mac
from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.utils.jppype import draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [3]:
PATH = Path("/home/gaby/Lab/DATA/Fundus/Fundus-AVSeg/")
RAW = PATH / "1-images"
AV = PATH / "2-av"
IMG = list(RAW.glob("*.png"))[0].name

fundus_gt = FundusData(image=RAW / IMG, av=AV / IMG).crop_to_roi(ensure_square=True).rescale(1500)


segment_od_mac(fundus_gt)
fundus = fundus_gt.copy()
_ = segment_av(fundus)

In [4]:
naive_av2tree = NaiveAVSegToTree()
av2tree = GNNAVSegToTree()

trees = av2tree(fundus)
trees_gt = naive_av2tree(fundus_gt)


m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=800, background=fundus.image)
fundus.draw(view=m[0])
draw_trees(trees, view=m[0])
fundus_gt.draw(view=m[1])
draw_trees(trees_gt, view=m[1])
m

[ WARN:0@10.174] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

## Compute AV topological maps


In [5]:
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import TopologicalLabel, rasterize_tree_topology

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10, bridge_gap_smaller_than=50) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=800,
    background=fundus.image,
)
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=False)
m[0, 0].add_label(fundus_gt.av == 1, colormap="red", opacity=0.2)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=False)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">VTree</h3>'), HTML(value=…

In [6]:
m = Mosaic(
    3,
    cell_height=400,
    background=fundus.image,
)
fundus_gt.draw(view=m[0])
draw_trees(trees, view=m[0], edge_labels=True)
m[1].add_image(TopologicalLabel.map_to_rgb(art_labels))
draw_tree(trees[0], view=m[1], artery=True)
m[2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
draw_tree(trees[1], view=m[2], artery=False)
m

GridBox(children=(View2D(linkedTransformGroup='a96b1e9ae2f84b8a9916058f82e81dca'), View2D(linkedTransformGroup…

In [7]:
import pandas as pd
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import evaluate_topology

df = pd.DataFrame(evaluate_topology(trees[0], art_labels, art_topo))

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_roots
from fundus_vessels_toolkit.segment_to_graph.models.training import deteriorate_segmentation
from fundus_vessels_toolkit.segment_to_graph.line_digraph_solving import prepare_graph_for_reconnections

av = deteriorate_segmentation(
    fundus_gt,
)
fundus_gt2 = fundus_gt.update(av=av)
trees_gt2 = naive_av2tree(fundus_gt2)
art_graph, candidates = prepare_graph_for_reconnections(
    trees_gt2[0], max_distance=300
)  # , branch_ids=[25], endpoint_ids=[84])
art_tree = naive_infer_roots(art_graph, fundus_gt2.od_center)

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=800)
fundus_gt2.draw(view=m[0])
draw_tree(art_tree, artery=True, view=m[0])

m[1].add_image(TopologicalLabel.map_to_rgb(art_labels))
draw_trees(trees_gt2, view=m[1])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [9]:
from fundus_vessels_toolkit.segment_to_graph.line_digraph_solving import prepare_graph_for_reconnections
from fundus_vessels_toolkit.utils.jppype import draw_graph

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=800)
fundus_gt.draw(view=m[0])
draw_tree(trees_gt2[0], edge="skeleton", artery=True, view=m[0], edge_labels=True)
fundus_gt.draw(view=m[1])
_, candidates = prepare_graph_for_reconnections(trees_gt2[0], max_distance=300)  # , branch_ids=[25], endpoint_ids=[84])
draw_graph(_, view=m[1], node_labels=True)
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [10]:
_.node_coord()[25]

array([569., 253.])

In [11]:
candidates[candidates[:, 1] == 107]

array([], shape=(0, 2), dtype=int64)

In [12]:
df_art = pd.DataFrame(evaluate_topology(trees_gt2[0], art_labels, art_topo))
df_vei = pd.DataFrame(evaluate_topology(trees_gt2[1], vei_labels, vei_topo))

In [13]:
df_art.iloc[48]

missing       0.000000
gt_dir        0.963928
gt_parent    15.000000
parent       15.000000
Name: 48, dtype: float64